# 04. Modelos tabulares

Este cuaderno entrena los modelos que usan atributos visuales globales de cada fotografia: el
predictor constante de referencia, una regresion lineal sobre el exceso de verde y un Random Forest
Regressor. Todos estiman la severidad del dano condicionada a sequia: se entrenan solo con los
registros DR del subconjunto de entrenamiento y, al evaluar, la regla del conjunto asigna 0 a los
diagnosticos distintos de DR. Los artefactos y resultados quedan en `data/procesado`, de donde los
leen la API y el cuaderno de comparacion.

## Entorno

In [ ]:
import sys
from pathlib import Path


def raiz_proyecto():
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "src").is_dir():
            return candidato
    raise RuntimeError("No se encontro la raiz del proyecto")


RAIZ = raiz_proyecto()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

%matplotlib inline
import numpy as np
import pandas as pd
from sklearn.base import clone

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

from src import graficos
from src.conjunto import cargar_conjunto
from src.evaluacion import cargar_resultados, metricas
from src.modelos_tabulares import (
    ATRIBUTOS_BOSQUE,
    RandomForestMagnitud,
    ajustar_bosque,
    ejecutar_random_forest,
    ejecutar_referencia_constante,
    ejecutar_regresion_exceso_verde,
    historial_busqueda,
)
from src.particiones import campos_compartidos, entrenamiento_y_prueba, separar_por_temporada

graficos.aplicar_estilo()

## Conjunto de modelado

El conjunto de modelado une los metadatos aptos con los atributos de cada imagen y trae fijada la
particion entre entrenamiento y prueba, agrupada por campo con semilla fija. El ajuste de
hiperparametros solo ve el entrenamiento DR; la prueba se usa una sola vez, al guardar los
resultados de cada modelo.

In [ ]:
conjunto = cargar_conjunto()
entrenamiento, prueba = entrenamiento_y_prueba(conjunto, solo_dr=True)

print(f"registros del conjunto: {len(conjunto)}")
print(f"DR de entrenamiento: {len(entrenamiento)} en {entrenamiento['id_campo'].nunique()} campos")
print(f"DR de prueba: {len(prueba)} en {prueba['id_campo'].nunique()} campos")

## Entrenamiento de los modelos

Cada funcion entrena, guarda el artefacto en `data/procesado/modelos` y guarda los resultados de
prueba en `data/procesado/resultados`. El Random Forest busca 30 combinaciones de hiperparametros
con `RandomizedSearchCV`, validando con cinco pliegues agrupados por campo para que un mismo campo
nunca quede a ambos lados de la validacion. La combinacion se elige por menor RMSE de validacion y
se reajusta con todo el entrenamiento DR. Su tiempo de entrenamiento incluye la busqueda completa y
ese reajuste.

In [ ]:
rutas = [
    ejecutar_referencia_constante(conjunto),
    ejecutar_regresion_exceso_verde(conjunto),
    ejecutar_random_forest(conjunto, n_iter=30),
]
for ruta in rutas:
    print(ruta.relative_to(RAIZ))

In [ ]:
resultados = {modelo_id: cargar_resultados(modelo_id) for modelo_id in ("constante", "regresion_exceso_verde", "random_forest")}
pd.DataFrame(
    [
        {
            "modelo": modelo_id,
            "hiperparametros": r.hiperparametros,
            "entrenamiento_s": round(r.tiempo_entrenamiento_s, 3),
            "inferencia_ms": round(r.tiempo_inferencia_ms, 4),
        }
        for modelo_id, r in resultados.items()
    ]
)

## Ajuste de hiperparametros del Random Forest

Cada fila es una combinacion probada, con el RMSE y el MAE medios de validacion sobre la magnitud
en escala de 0 a 100. La fila marcada como elegida es la que se reajusto y se evaluo en la prueba.

In [ ]:
resultados_rf = resultados["random_forest"]
historial = pd.DataFrame(
    [
        {**paso.parametros, "rmse_validacion": paso.rmse_validacion, "mae_validacion": paso.mae_validacion, "elegido": paso.elegido}
        for paso in resultados_rf.historial_ajuste
    ]
).sort_values("rmse_validacion", ignore_index=True)
historial.round(3)

## Atributos de dispositivo

El analisis de imagenes mostro que `megapixeles` y `nitidez` cambian con la temporada: la resolucion
media pasa de 0.49 megapixeles en las primeras temporadas a 3.39 en SR2021. Estos atributos pueden
describir el telefono o la temporada mas que el dano, por lo que el modelo final los excluye. Para
cuantificar esa decision se repite la busqueda agregandolos y se comparan ambas variantes solo con
la validacion por campo, sin tocar la prueba ni guardar resultados.

In [ ]:
elegido = next(paso for paso in resultados_rf.historial_ajuste if paso.elegido)
busqueda_dispositivo = ajustar_bosque(entrenamiento, ATRIBUTOS_BOSQUE + ("megapixeles", "nitidez"), n_iter=30)
elegido_dispositivo = next(paso for paso in historial_busqueda(busqueda_dispositivo) if paso["elegido"])

pd.DataFrame(
    [
        {"variante": "sin megapixeles ni nitidez", "atributos": len(ATRIBUTOS_BOSQUE), "rmse_validacion": elegido.rmse_validacion, "mae_validacion": elegido.mae_validacion},
        {"variante": "con megapixeles y nitidez", "atributos": len(ATRIBUTOS_BOSQUE) + 2, "rmse_validacion": elegido_dispositivo["rmse_validacion"], "mae_validacion": elegido_dispositivo["mae_validacion"]},
    ]
).round(3)

## Importancia de los atributos

La importancia de cada atributo es la reduccion media de impureza que aporta en los arboles del
modelo elegido, normalizada para que la suma sea 1.

In [ ]:
importancias = pd.DataFrame([i.model_dump() for i in resultados_rf.importancias])
importancias["importancia"] = importancias["importancia"].round(3)
fig = graficos.barras_frecuencia(importancias, "Importancia de los atributos en Random Forest", columna_valor="importancia")
for etiqueta in fig.axes[0].get_xticklabels():
    etiqueta.set(rotation=45, horizontalalignment="right")
graficos.guardar_figura(fig, "04_importancia_atributos")
importancias

## Generalizacion temporal

La particion por campo mezcla las cuatro temporadas en ambos lados. Para medir la generalizacion
hacia una temporada nueva, dentro del entrenamiento DR se entrena el Random Forest con los
hiperparametros elegidos sobre las temporadas anteriores y se evalua en SR2021. Se usa solo el
subconjunto de entrenamiento para no gastar la prueba. Tambien se cuentan los campos que aparecen en
ambas partes, porque un campo compartido puede inflar el resultado.

In [ ]:
anteriores, sr2021 = separar_por_temporada(entrenamiento)
elegido_rf = RandomForestMagnitud()
elegido_rf.cargar()
temporal = RandomForestMagnitud().entrenar(anteriores, clone(elegido_rf.estimador))
valores = metricas(sr2021["magnitud"], np.clip(temporal.predecir_lote(sr2021), 0, 100))

print(f"entrenamiento sin SR2021: {len(anteriores)} registros DR")
print(f"evaluacion en SR2021: {len(sr2021)} registros DR")
print(f"campos compartidos entre ambas partes: {len(campos_compartidos(anteriores, sr2021))}")
pd.DataFrame(
    [
        {"evaluacion": "validacion por campo, 5 pliegues", "rmse": elegido.rmse_validacion, "mae": elegido.mae_validacion},
        {"evaluacion": "temporal, evaluado en SR2021", "rmse": valores["rmse"], "mae": valores["mae"]},
    ]
).round(3)